# Creating a ground truth by using label transfer from one annotated slide to the other slides. 

In [2]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.neighbors import KNeighborsClassifier
import gc

### Label transfer model

In [ ]:
# Define paths and exclusions
basedir = Path("../data")
ground_truth_dir = basedir / "CRC_P2/ground_truth"
spatial_dir_template = "{}/output/binned_outputs/square_002um/"
f_feature_matrix = "filtered_feature_bc_matrix.h5"
f_tissue_positions = "tissue_positions.parquet"
exclude_dirs = {"combined", "new", "CRC_P2"}

# =========================================================================
# 1. Load Ground Truth Reference Data
# =========================================================================
print("Loading reference ground truth data (CRC_P2)...")

# Load reference expression matrix
gt_adata = sc.read_10x_h5(
    basedir / spatial_dir_template.format("CRC_P2") / f_feature_matrix
)
gt_adata.var_names_make_unique()

# Load annotations CSV (tab-separated barcode -> annotation)
gt_csv_path = ground_truth_dir / "2um_squares_annotation.csv"
gt_df = pd.read_csv(gt_csv_path, sep="\t", header=None, names=["barcode", "annotation"])
gt_df = gt_df.set_index("barcode")

# Join ground truth annotations to AnnData obs
gt_adata.obs = gt_adata.obs.join(gt_df, how="left")

# Load spatial coordinates for reference
gt_tissue_pos = pd.read_parquet(
    basedir / spatial_dir_template.format("CRC_P2") / "spatial" / f_tissue_positions
)
if "barcode" in gt_tissue_pos.columns:
    gt_tissue_pos = gt_tissue_pos.set_index("barcode")

spatial_cols = ["pxl_row_in_fullres", "pxl_col_in_fullres"]
gt_adata.obs = gt_adata.obs.join(gt_tissue_pos[spatial_cols], how="left")

# Save pixel coordinates to spatial obsm array for standard scanpy compatibility
gt_adata.obsm["spatial"] = gt_adata.obs[spatial_cols].values

# Filter unannotated bins in reference if any exist
gt_adata = gt_adata[gt_adata.obs["annotation"].notna()].copy()

# Preprocess Reference AnnData
print("Preprocessing reference sample...")
sc.pp.normalize_total(gt_adata, target_sum=1e4)
sc.pp.log1p(gt_adata)
sc.pp.highly_variable_genes(gt_adata, n_top_genes=2000)
gt_adata_hvg = gt_adata[:, gt_adata.var["highly_variable"]].copy()

sc.tl.pca(gt_adata_hvg, svd_solver="arpack")
sc.pp.neighbors(gt_adata_hvg, n_neighbors=15)
sc.tl.umap(gt_adata_hvg)

# Train a KNN classifier for confidence scoring
knn_model = KNeighborsClassifier(n_neighbors=15, weights="distance")
knn_model.fit(gt_adata_hvg.obsm["X_pca"], gt_adata_hvg.obs["annotation"])


# =========================================================================
# 2. Process Target Samples & Perform Label Transfer
# =========================================================================
# Identify all valid sample subdirectories in basedir
all_samples = [
    d.name for d in basedir.iterdir() 
    if d.is_dir() and d.name not in exclude_dirs
]

target_adatas = {}

for sample_id in all_samples:
    sample_dir = basedir / spatial_dir_template.format(sample_id)
    matrix_file = sample_dir / f_feature_matrix
    pos_file = sample_dir / "spatial" / f_tissue_positions
    
    if not matrix_file.exists() or not pos_file.exists():
        print(f"Skipping {sample_id}: Required files missing.")
        continue
        
    print(f"\n--- Processing Target Sample: {sample_id} ---")
    
    # 1. Load single target sample into RAM
    target_adata = sc.read_10x_h5(matrix_file)
    target_adata.var_names_make_unique()
    
    # 2. Join tissue pixel coordinates
    tissue_pos = pd.read_parquet(pos_file)
    if "barcode" in tissue_pos.columns:
        tissue_pos = tissue_pos.set_index("barcode")
        
    target_adata.obs = target_adata.obs.join(tissue_pos[spatial_cols], how="left")
    target_adata.obsm["spatial"] = target_adata.obs[spatial_cols].values
    
    # 3. Preprocess
    sc.pp.normalize_total(target_adata, target_sum=1e4)
    sc.pp.log1p(target_adata)
    
    # 4. Subset to reference HVGs
    common_genes = gt_adata_hvg.var_names.intersection(target_adata.var_names, sort=False)
    gt_adata_common = gt_adata_hvg[:, common_genes].copy()
    target_adata_hvg = target_adata[:, common_genes].copy()
    
    # 5. Transfer Labels & Confidence
    sc.tl.ingest(target_adata_hvg, gt_adata_hvg, obs="annotation")
    target_adata.obs["predicted_annotation"] = target_adata_hvg.obs["annotation"]
    
    probs = knn_model.predict_proba(target_adata_hvg.obsm["X_pca"])
    target_adata.obs["predicted_annotation_probability"] = probs.max(axis=1)
    
    target_adata.obs["predicted_annotation_filtered"] = target_adata.obs["predicted_annotation"].copy()
    target_adata.obs.loc[
        target_adata.obs["predicted_annotation_probability"] < 0.4, 
        "predicted_annotation_filtered"
    ] = "Unassigned / Low Confidence"
    
    # 6. Save back to the sample's own directory
    out_file = sample_dir / "at_filtered_feature_bc_matrix.h5ad"
    target_adata.write_h5ad(out_file)
    print(f"Saved directly to source directory: {out_file}")
    
    # 7. Clear memory for the next sample
    del target_adata, target_adata_hvg, tissue_pos, probs

print("\nProcessing complete! All target samples have been saved alongside their raw matrices.")

Loading reference ground truth data (CRC_P2)...


/home/charlottewolf/miniconda3/envs/autoencoder/lib/python3.12/site-packages/scanpy/readwrite.py:248: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata = adata.copy()


Preprocessing reference sample...


/tmp/ipykernel_583809/402412802.py:46: UserWarning: Some cells have zero counts
  sc.pp.normalize_total(gt_adata, target_sum=1e4)
